In [130]:
import dataclasses
import json
import pandas as pd


@dataclasses.dataclass
class RideHomework:
    lpep_pickup_datetime: int
    lpep_dropoff_datetime: int
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float

    def rides() -> pd.DataFrame:
        """Read dataset and return cleaned dataframe"""

        URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

        columns = {
            "lpep_pickup_datetime": "datetime64[ns]",
            "lpep_dropoff_datetime": "datetime64[ns]",
            "PULocationID": "int32",
            "DOLocationID": "int32",
            "passenger_count": "Int64",
            "trip_distance": "float32",
            "tip_amount": "float32",
            "total_amount": "float32",
        }

        df = pd.DataFrame()
        try:
            df = pd.read_parquet(
                URL,
                columns=list(columns),
            ).astype(columns)

            df["passenger_count"] = df["passenger_count"].astype("Int64")
            df["lpep_pickup_datetime"] = df["lpep_pickup_datetime"].astype("int64")
            df["lpep_dropoff_datetime"] = df["lpep_dropoff_datetime"].astype("int64")

            display(df.head())
            # display(df.info())
            display(df.shape)

            return df
        except ImportError as e:
            print(f"[ERROR]: {e}")
            print("Please install pyarrow or fastparquet to enable Parquet support.")
            raise
        except Exception as e:
            print(f"[ERROR] An error occurred: {e}")
            raise

    def from_record(row):
        """Returns a Ride object from a record"""
        return RideHomework(**row)

    def value_serializer(ride):
        """Serializes Ride object to bytes"""
        ride_dict = dataclasses.asdict(ride)
        json_str = json.dumps(ride_dict)
        return json_str.encode("utf-8")

    def value_deserializer(msg):
        """Deserializes bytes to a Ride object"""
        msg_str = msg.decode("utf-8")
        ride_dict = json.loads(msg_str)
        return RideHomework(**ride_dict)

    def value_json_deserializer(json_data):
        """Deserializes json to a Ride object"""
        return RideHomework(**json_data)

In [131]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=["localhost:19092"],
    request_timeout_ms=2000,
    api_version_auto_timeout_ms=2000,
    value_serializer=RideHomework.value_serializer,
)

In [132]:
def produce():
    print("[INFO] Starting Producer APP.")

    topic = "green-trips"

    rides: pd.DataFrame = RideHomework.rides()
    for record in rides.to_dict(orient="records"):
        ride = RideHomework.from_record(record)
        producer.send(
            topic=topic,
            value=ride,
        )

    print("[INFO] Stop producer APP")

In [133]:
from time import time

t0 = time()

produce()
producer.flush()
producer.close()

t1 = time()

print(f"[INFO] Publish green trips took {(t1 - t0):.2f} seconds")

[INFO] Starting Producer APP.


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,1759278107000000000,1759278277000000000,247,69,1,0.70,1.70,10.000000
1,1759277643000000000,1759278254000000000,66,25,1,1.61,2.78,16.680000
2,1759277804000000000,1759277807000000000,244,244,1,0.00,2.20,13.200000
3,1759277256000000000,1759278734000000000,95,170,1,10.37,11.31,67.849998
4,1759266629000000000,1759267350000000000,82,138,1,4.07,6.82,34.119999


(49416, 8)

KeyboardInterrupt: 

In [ ]:
from kafka import KafkaConsumer

print("[INFO] Starting consumer APP")
print("[INFO] Type CTRL + C to stop the consumer APP")

topic = "green-trips"

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=["localhost:19092"],
    client_id="consumer_test_1",
    group_id="consumer_test_group",
    auto_offset_reset="earliest",
    value_deserializer=RideHomework.value_deserializer,
)

try:
    trips_distance_more_than_5 = 0
    for message in consumer:
        ride = message.value
        if ride.trip_distance > 5.0:
            trips_distance_more_than_5 += 1
except KeyboardInterrupt:
    print("\n[INFO] KeyboardInterrupt received, stopping consumer...")

finally:
    print(f"[INFO] Trips longer than 5 kilometers count: {trips_distance_more_than_5}")
    consumer.close()  # ensure the consumer shuts down cleanly
    print("[INFO] Consumer APP stopped")